<a href="https://colab.research.google.com/github/wavymejti/KursAI1/blob/main/titanic2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [17]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.preprocessing import StandardScaler
import numpy as np

In [18]:
#wczytanie i zapoznanie się z danymi
df = sns.load_dataset("titanic")
print("10 pierwszych wierszow z datasetu titanic\n\n", df.head(10))

10 pierwszych wierszow z datasetu titanic

    survived  pclass     sex   age  sibsp  parch     fare embarked   class  \
0         0       3    male  22.0      1      0   7.2500        S   Third   
1         1       1  female  38.0      1      0  71.2833        C   First   
2         1       3  female  26.0      0      0   7.9250        S   Third   
3         1       1  female  35.0      1      0  53.1000        S   First   
4         0       3    male  35.0      0      0   8.0500        S   Third   
5         0       3    male   NaN      0      0   8.4583        Q   Third   
6         0       1    male  54.0      0      0  51.8625        S   First   
7         0       3    male   2.0      3      1  21.0750        S   Third   
8         1       3  female  27.0      0      2  11.1333        S   Third   
9         1       2  female  14.0      1      0  30.0708        C  Second   

     who  adult_male deck  embark_town alive  alone  
0    man        True  NaN  Southampton    no  False  


In [19]:
#Wyświetlenie informacji o brakujących wartościach
print("\nLiczba brakujących danych w zestawie przed czyszczeniem")
print(df.isnull().sum())


Liczba brakujących danych w zestawie przed czyszczeniem
survived         0
pclass           0
sex              0
age            177
sibsp            0
parch            0
fare             0
embarked         2
class            0
who              0
adult_male       0
deck           688
embark_town      2
alive            0
alone            0
dtype: int64


In [20]:
df = df.dropna(subset=['age'])
print("\nLiczba brakujacych wartosci po czyszczeniu (tylko age)")
print(df.isnull().sum())

print(f"\nPrzed:{len(sns.load_dataset("titanic"))}\nPo:{len(df)}")


Liczba brakujacych wartosci po czyszczeniu (tylko age)
survived         0
pclass           0
sex              0
age              0
sibsp            0
parch            0
fare             0
embarked         2
class            0
who              0
adult_male       0
deck           530
embark_town      2
alive            0
alone            0
dtype: int64

Przed:891
Po:714


● niska (od najniższej ceny do najniższej ceny + 1/3 rozbieżności
cenowej)

● średnia (od najniższej ceny + 1/3 rozbieżności cenowej do
najniższej ceny + 2/3 rozbieżności cenowej )

● wysoka (od najniższej ceny + 2/3 rozbieżności cenowej do
najwyższej ceny)

In [21]:
def kategoryzacja_oplaty(fare, fare_ranges):
  if fare <= fare_ranges[0]:
    return 0 #niska
  elif fare <= fare_ranges[1]:
    return 1 #srednia
  else:
    return 2 #wysoka

min_fare = df['fare'].min()
max_fare = df['fare'].max()
fare_step = (max_fare - min_fare) / 3
fare_range = [min_fare + fare_step, min_fare + 2 * fare_step]

print("\nPrzedzialy cenowe biletów")
print(f"\nNiski: {min_fare:.2f} - {fare_range[0]:.2f}")
print(f"\nSredni: {fare_range[0]:.2f} - {fare_range[1]:.2f}")
print(f"\nWysoki: {fare_range[1]:.2f} - {max_fare:.2f}")


Przedzialy cenowe biletów

Niski: 0.00 - 170.78

Sredni: 170.78 - 341.55

Wysoki: 341.55 - 512.33


In [22]:
#przygotowanie danych do modelowania
selected_features = ['sex', 'pclass', 'age', 'fare_category']
df_model = df.copy()
df_model['sex'] = df_model['sex'].map({'male' : 0, 'female' : 1})
df_model['fare_category'] = df_model['fare'].apply(lambda x: kategoryzacja_oplaty(x, fare_range))




In [23]:
#czym jest funkcja anonimowa
def kwadratNormalny(x):
  return x**2

kwadrat = lambda x: x ** 2

print(f'normalna funkcja "kwadrat": {kwadratNormalny(5)}')
print(f'funkcja anonimowa (lambda): {kwadrat(5)}')

normalna funkcja "kwadrat": 25
funkcja anonimowa (lambda): 25


In [24]:
#podzial danych i skalowanie
#przygotowanie danych
X = df_model[selected_features]
y = df_model['survived']

#podział na zbiór treningowy i testowy
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

#skalowanie danych
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)